# Method contrast — PTO vs GRPO at matched look-ahead  `[EVAL]`

**RQ-ii: does iterative GRPO compete with PTO under a matched look-ahead depth?** The method contrast
as persona-paired tables at each K, on **both graders side by side** (column `judge`: `gpt-4o-mini` =
the training oracle, `claude-haiku-4-5` = the held-out judge). Everything is full-conversation eval,
paired by the 96 shared personas; the K=5 GRPO arm stops at iteration 5, so every K=5 row is
right-censored there. Exports → `results/method/contrast/{tables,figures}/` (judge-invariant family:
no `<judge>/` level — the grader is a column, never a folder). Ported from `7_Stats` §4a/§4b; the
same rows for the same arm and grader match the retired `results/L0|L5/tables/7_stats/<judge>/` tables.

**Sign convention:** `+ mean_delta ⇒ PTO higher`. On `MICI` (MI-inconsistent behaviour, lower = better)
a positive Δ therefore means PTO is *worse*; on every other rubric it means PTO is better.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting, stats
from eda_analysis.constants import judge_dirname, PRIMARY_JUDGE_TAG, LOWER_IS_BETTER
cfg = eda_analysis.EdaConfig(family="method/contrast", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp the banner reset_results just removed (judge-invariant: lives directly under figures/)

## 0 · Confirmatory vs exploratory — read this first  `[EVAL]`

**Confirmatory (a thesis claim).** *PTO > GRPO on Q1+Q2 at matched budget* — reported at **both** the
matched iteration (§1, `method_paired_by_K`: every iteration both arms reached) **and** as the
**best-vs-best model-selection contrast** (§2, `method_paired_best`: PTO at its own-oracle best iteration vs
GRPO at *its* best), so GRPO is credited at its peak rather than only at its regressed endpoint. The
held-out grader is the out-of-sample check on the same claim: it never scored a training reward, so a
gap that survives it is not the oracle grading its own homework.

**Multiplicity scope.** Each row's `p_holm` is Holm-corrected across the rubrics **within its own
(judge, K, iteration) contrast** — every matched-budget point is its own family; corrections are **not**
pooled across iterations or graders. `mean_delta` / `dz` are unaffected by the correction.

**Exploratory.** Every non-`Q1Q2` rubric here (WAI-SR, CSQ-8, MI-SAT, MITI, PCT, MICI, Q1, Q2) is
hypothesis-generating; the K=5 rows are matched on *iteration*, not *budget* — the budget-matched
form of the same contrast lives in `compute/cost` (§4 pointer).

In [ ]:
SC = eda_analysis.scores_by_judge(S)          # {judge_label: scores_long}, primary FIRST
PRIMARY = judge_dirname(PRIMARY_JUDGE_TAG)
JUDGES = list(SC)
ROLE = {j: ("training oracle" if j == PRIMARY else "held-out judge") for j in JUDGES}
for j, sc in SC.items():
    print(f"{j:18s} ({ROLE[j]:15s}) scores_long {sc.shape} | arms {sorted(sc.arm.unique())} | "
          f"K {sorted(sc.K.unique())}")
KS = sorted(SC[PRIMARY].K.unique())
print("K levels:", KS)

## 1 · Matched iteration — PTO − GRPO at each K, every iteration both arms reached  `[EVAL]`
**Purpose.** The method contrast as a paired table at matched iterations, per grader (iteration-0 base
rows dropped from the table — the two arms' bases are independent draws of the same model, and are
kept only as the noise floor in the §3 figure). Columns: `judge`, `K`, `iteration`, `metric`, `n`,
`mean_delta`, `dz`, `p_holm`. Persona-paired Wilcoxon signed-rank + Cohen's *dz* + Holm across rubrics
within each (judge, K, iteration). At K=5 the table ends at GRPO_LA5's last iteration (5); at K=0 both
arms run to 10.

In [ ]:
frames = []
for j, sc in SC.items():
    for K in sorted(sc.K.unique()):
        CMP = stats.paired_method_comparison(sc, "PTO", "GRPO", K=int(K))
        if not CMP.empty:
            frames.append(CMP.assign(judge=j))
MP = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if MP.empty:
    print("no common PTO/GRPO iterations at any K under any grader.")
else:
    MP["judge"] = pd.Categorical(MP["judge"], categories=JUDGES, ordered=True)
    MP = MP.sort_values(["judge", "K", "iteration"], kind="stable").reset_index(drop=True)
    MP["judge"] = MP["judge"].astype(str)
    MPt = MP[MP.iteration > 0]
    view = MPt[["judge", "K", "iteration", "metric", "n", "mean_delta", "dz", "p_holm"]].round(4)
    print("=== PTO - GRPO at matched K + iterations, per grader (+ => PTO higher) ===")
    display(view[view.metric.isin(["Q1Q2", "MICI"])])
    exports.save_table(view, "method_paired_by_K", caption=(
        "PTO - GRPO at matched K and matched iterations, BOTH graders in one table (column `judge`: "
        "gpt-4o-mini = the training oracle, claude-haiku-4-5 = the held-out judge; column `K` = look-ahead "
        "depth). Persona-paired (n = 96 shared personas) Wilcoxon + Cohen's dz + Holm. + => PTO higher; on "
        "MICI (lower = better) a positive delta means PTO is WORSE. Holm scope: p_holm is corrected across "
        "the rubrics WITHIN each (judge, K, iteration) contrast, NOT across iterations or graders (each "
        "matched-budget point is its own family). K=5 rows stop at iteration 5 (GRPO_LA5 right-censored "
        "there); K=0 runs to 10. Matched on ITERATION, not budget - see compute/cost for the budget-matched "
        "method sweeps. Same statistic as the retired results/L0|L5/tables/7_stats/<judge>/method_paired_by_K."))

## 2 · Best-vs-best — the model-selection contrast  `[EVAL]`
**Purpose.** PTO at its **own-oracle best** iteration vs GRPO at **its** best (per `best_per_experiment`,
selected under *that grader's* scores — so the held-out row selects checkpoints by the held-out grader),
persona-paired across the different iterations (valid: every iteration reshuffles the same 96 personas).
This is the strongest-steelman comparison: GRPO is credited at its peak, before the post-peak regression.
Complements the matched-iteration table above; `iter_a` = PTO's selected iteration, `iter_b` = GRPO's.
At K=5 the GRPO side can only be selected from iterations 1–5 (censored).

In [ ]:
frames = []
for j, sc in SC.items():
    for K in sorted(sc.K.unique()):
        BB = stats.paired_best_method_comparison(sc, "PTO", "GRPO", K=int(K))
        if not BB.empty:
            frames.append(BB.assign(judge=j))
BBall = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if BBall.empty:
    print("no K with both PTO and GRPO best models scored under any grader.")
else:
    BBall["judge"] = pd.Categorical(BBall["judge"], categories=JUDGES, ordered=True)
    BBall = BBall.sort_values(["judge", "K"], kind="stable").reset_index(drop=True)
    BBall["judge"] = BBall["judge"].astype(str)
    view_bb = BBall[["judge", "K", "iter_a", "iter_b", "metric", "n", "mean_delta", "dz", "p_holm"]].round(4)
    print("=== PTO(best) - GRPO(best) per K and grader (+ => PTO higher; iter_a = PTO best, iter_b = GRPO best) ===")
    display(view_bb)
    exports.save_table(view_bb, "method_paired_best", caption=(
        "PTO at its own-oracle BEST iteration vs GRPO at ITS best, per K, BOTH graders in one table (column "
        "`judge`: gpt-4o-mini = the training oracle, claude-haiku-4-5 = the held-out judge; each grader's "
        "row selects the checkpoints under ITS OWN scores). Persona-paired (n = 96) Wilcoxon + dz + Holm "
        "across rubrics within each (judge, K). + => PTO higher; on MICI (lower = better) a positive delta "
        "means PTO is WORSE. The model-selection contrast: GRPO credited at its peak (iter_b), before the "
        "post-peak regression; complements method_paired_by_K. At K=5 GRPO's best is chosen from iterations "
        "1-5 only (GRPO_LA5 right-censored at 5), so the K=5 rows are descriptive. Same statistic as the "
        "retired results/L0|L5/tables/7_stats/<judge>/method_paired_best."))

## 3 · The method gap by iteration — one panel per grader  `[EVAL]`
**Purpose.** §1's `Q1Q2` rows as a picture: PTO − GRPO per iteration at K=0 (black, solid, circles) and
K=5 (green, dashed, squares), one panel per grader, ribbons = persona-bootstrap 95% CI, stars = cleared
Holm (across rubrics within that (judge, K, iteration) — the same `p_holm` as the table). Iteration 0 is
the two arms' independent base draws (hollow marker) and is the noise floor. The K=5 line stops at
iteration 5 (GRPO_LA5 censored). Rendered inline (the style of `plotting.k_did`'s method-gap row) from the
§1 frame, so the figure never disagrees with the table it sits next to.

In [ ]:
def method_gap_fig(MP, metric="Q1Q2", judges=None, gap_colors=None):
    # One column per grader; the K=0 / K=5 gap PTO - GRPO with CI ribbons + Holm stars.
    gap_colors = gap_colors or {0: "#111111", 5: "#009E73"}
    K_STYLE = plotting.lookahead.K_STYLE
    judges = judges or list(dict.fromkeys(MP["judge"]))
    sub_all = MP[MP.metric == metric]
    if sub_all.empty:
        return None
    n_it = int(sub_all.iteration.max()) + 1
    fig, axes = plt.subplots(1, len(judges), figsize=(3.6 * len(judges), 3.3), sharey=True, squeeze=False)
    for j, jn in enumerate(judges):
        ax = axes[0, j]
        for K in sorted(sub_all.K.unique()):
            sub = sub_all[(sub_all.judge == jn) & (sub_all.K == K)].sort_values("iteration")
            if sub.empty:
                continue
            st = K_STYLE.get(int(K), {"ls": "-", "marker": "o"})
            col = gap_colors.get(int(K), "0.4")
            ax.fill_between(sub.iteration, sub.ci_low, sub.ci_high, color=col, alpha=0.13, lw=0)
            pos = sub[sub.iteration > 0]
            ax.plot(pos.iteration, pos.mean_delta, ls=st["ls"], marker=st["marker"], color=col, lw=1.7, ms=5.5)
            base = sub[sub.iteration == 0]
            if not base.empty:            # the two independent base draws: hollow marker + a dotted link
                ax.plot(sub.iteration.iloc[:2], sub.mean_delta.iloc[:2], ls=":", color=col, lw=1.0)
                ax.scatter(base.iteration, base.mean_delta, marker=st["marker"], s=34, facecolor="white",
                           edgecolor=col, linewidth=1.3, zorder=4)
            sig = pos[pos.p_holm < 0.05]
            ax.scatter(sig.iteration, sig.mean_delta, marker="*", s=80, color=col, zorder=5,
                       edgecolor="white", linewidth=0.6)
        ax.axhline(0, color="0.35", lw=0.8)
        ax.set_title(f"{jn} ({ROLE.get(jn, '')})", fontsize=9.5)
        ax.set_xlabel("iteration (0 = the two arms' base draws)", fontsize=9)
        ax.set_xticks(range(0, n_it))
        ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.35)
    axes[0, 0].set_ylabel(f"{metric} Δ (PTO − GRPO), score points", fontsize=9)
    k5_end = int(sub_all.loc[sub_all.K == 5, "iteration"].max()) if (sub_all.K == 5).any() else None
    h = [Line2D([], [], color=gap_colors[0], ls="-", marker="o", ms=5.5, lw=1.7, label="K=0: PTO_LA0 − GRPO_LA0"),
         Line2D([], [], color=gap_colors[5], ls="--", marker="s", ms=5.5, lw=1.7,
                label="K=5: PTO_LA5 − GRPO_LA5" + (f" (to iter {k5_end}, GRPO_LA5 censored)" if k5_end is not None else "")),
         Line2D([], [], color="0.2", ls="none", marker="*", ms=9, mec="white", mew=0.5,
                label="Holm p < .05 (across rubrics within the (judge, K, iteration) contrast)")]
    lg = fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False, fontsize=8,
                    title=f"{metric} gap PTO − GRPO by iteration, one panel per grader.  + => PTO higher.  "
                          "Ribbons = persona-bootstrap 95% CI, n = 96 personas.", title_fontsize=7.5)
    lg.get_title().set_color("0.3")
    fig.tight_layout()
    return fig

if not MP.empty:
    fig = method_gap_fig(MP, metric="Q1Q2", judges=JUDGES)
    if fig is not None:
        exports.save_fig(fig, "method_gap", caption=(
            "The method gap PTO - GRPO on Q1+Q2 per iteration, one panel per grader (left gpt-4o-mini = the "
            "training oracle, right claude-haiku-4-5 = the held-out judge): K=0 black solid circles, K=5 green "
            "dashed squares, ribbons = persona-bootstrap 95% CI over the 96 shared personas, stars = cleared "
            "Holm across rubrics within that (judge, K, iteration) contrast (the p_holm of method_paired_by_K). "
            "+ => PTO higher. Iteration 0 (hollow) is the two arms' independent base draws of the same model = "
            "the noise floor. The K=5 line stops at iteration 5 (GRPO_LA5 right-censored there); matched on "
            "iteration, not budget."))
        plt.show()

### 3b · Number ledger  `[EVAL]`
The headline cells of §1/§2 as citable keys (`results/method/contrast/tables/method_contrast.json`): for each
grader × K, the Q1+Q2 gap at the last matched iteration and at best-vs-best. Every value is a cell of the
two tables above (the `source` field says which); nothing is computed here that is not in a table.

In [ ]:
if not MP.empty:
    NUM = {}
    for j in JUDGES:
        for K in KS:
            m = MPt[(MPt.judge == j) & (MPt.K == K) & (MPt.metric == "Q1Q2")]
            if not m.empty:
                last = m[m.iteration == m.iteration.max()].iloc[0]
                pre = f"matched_last.{j}.K{int(K)}.Q1Q2"
                src = "tables/method_paired_by_K.md"
                note = f"PTO_LA{int(K)} - GRPO_LA{int(K)} at the last matched iteration ({int(last.iteration)}); + => PTO higher"
                NUM[f"{pre}.iteration"]  = {"value": int(last.iteration), "source": src, "note": note}
                NUM[f"{pre}.mean_delta"] = {"value": float(last.mean_delta), "source": src, "note": note}
                NUM[f"{pre}.dz"]         = {"value": float(last.dz), "source": src, "note": note}
                NUM[f"{pre}.p_holm"]     = {"value": float(last.p_holm), "source": src, "note": note}
                NUM[f"{pre}.n"]          = {"value": int(last.n), "source": src, "note": note}
            if not BBall.empty:
                b = BBall[(BBall.judge == j) & (BBall.K == K) & (BBall.metric == "Q1Q2")]
                if not b.empty:
                    b = b.iloc[0]
                    pre = f"best_vs_best.{j}.K{int(K)}.Q1Q2"
                    src = "tables/method_paired_best.md"
                    note = (f"PTO_LA{int(K)} at its best iteration ({int(b.iter_a)}) - GRPO_LA{int(K)} at its best "
                            f"({int(b.iter_b)}), each selected under this grader; + => PTO higher")
                    NUM[f"{pre}.iter_pto"]   = {"value": int(b.iter_a), "source": src, "note": note}
                    NUM[f"{pre}.iter_grpo"]  = {"value": int(b.iter_b), "source": src, "note": note}
                    NUM[f"{pre}.mean_delta"] = {"value": float(b.mean_delta), "source": src, "note": note}
                    NUM[f"{pre}.dz"]         = {"value": float(b.dz), "source": src, "note": note}
                    NUM[f"{pre}.p_holm"]     = {"value": float(b.p_holm), "source": src, "note": note}
                    NUM[f"{pre}.n"]          = {"value": int(b.n), "source": src, "note": note}
    NUM["meta.sign"] = {"value": "+ => PTO higher (on MICI, lower = better, + => PTO worse)", "source": "", "note": ""}
    NUM["meta.pairing"] = {"value": "persona-paired, n = 96 shared personas per contrast", "source": "", "note": ""}
    NUM["meta.censoring"] = {"value": "GRPO_LA5 right-censored at iteration 5; K=5 contrasts end there", "source": "", "note": ""}
    NUM["meta.holm_scope"] = {"value": "across rubrics within each (judge, K, iteration) contrast", "source": "", "note": ""}
    path = exports.save_numbers("method_contrast", NUM, caption=(
        "Number ledger for the method contrast: the Q1+Q2 gap PTO - GRPO at the last matched iteration "
        "(matched_last.*) and at best-vs-best (best_vs_best.*), per grader x K, each key citing the table "
        "cell it was read from. + => PTO higher; persona-paired n = 96; GRPO_LA5 censored at iteration 5."))
    print(f"{len(NUM)} keys ->", path)
    display(pd.DataFrame([{"key": k, **v} for k, v in NUM.items() if not k.startswith("meta.")])
            .drop(columns=["note"]).set_index("key"))

## 4 · Where the budget-matched method contrast lives  `[EVAL]`
Both tables above match on **iteration**, which is not a fixed unit of spend: a whole PTO iteration costs a
fraction of a GRPO one (its dominant phase is the preference-tree *build*, and it needs no in-loop
reward computation), and a K=5 step costs ~1.9× a K=0 step. The same contrast at matched **GPU-hours** —
`method_K0` (PTO_LA0 vs GRPO_LA0) and `method_K5` (PTO_LA5 vs GRPO_LA5), each arm represented by the best
checkpoint reachable within a budget, on both graders and both selection metrics — is owned by
**`compute/cost`** (`compute.all_budget_sweeps` / `budget_sweep_crossjudge`; tables
`results/compute/cost/tables/budget_sweep_method_K{0,5}_*` and the `budget_sweep_crossjudge*` verdicts,
figure `budget_sweep_grid`). Quote the *sweep*, not a single iso-compute row: the sign of a lever is a
function of budget. This family deliberately does not recompute it — one owner per fact.

In [ ]:
_cost_tables = os.path.join(exports.RESULTS_DIR, "compute", "cost", "tables")
_hits = sorted(f for f in (os.listdir(_cost_tables) if os.path.isdir(_cost_tables) else [])
               if "method" in f and f.endswith(".md"))
if _hits:
    print("budget-matched method contrast tables rendered by compute/cost:")
    for f in _hits: print("   results/compute/cost/tables/" + f)
else:
    print("compute/cost has not rendered yet — run tools/render_results.py --family compute/cost "
          "for the budget-matched method sweeps.")

## 5 · Artifact index
Drop captions whose artifact no longer exists, then refresh `results/method/INDEX.md` + `results/INDEX.md`.

In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())